In [2]:
!pip install -U langchain langchain_openai langchain_community requests python-dotenv beautifulsoup4

In [3]:
import os

os.environ["OPENROUTER_API_KEY"] = "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"


In [4]:
import os

print(os.getenv("OPENROUTER_API_KEY")[:12])

sk-or-v1-e48


In [10]:
os.environ["TAVILY_API_KEY"] = "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"


In [11]:
import os
import requests
from bs4 import BeautifulSoup

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

from langchain_community.tools.tavily_search import TavilySearchResults

In [12]:
internet_search = TavilySearchResults()

/tmp/ipykernel_575/3012731111.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  internet_search = TavilySearchResults()


In [13]:
!pip install -U langchain-tavily

In [16]:
from langchain_tavily import TavilySearch

internet_search = TavilySearch(max_results=3)

In [15]:
internet_search.invoke({"query": "AI in education"})

{'query': 'AI in education',
 'response_time': 1.15,
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.unesco.org/en/digital-education/artificial-intelligence',
   'title': 'Artificial intelligence in education - AI | UNESCO',
   'content': '# Artificial intelligence in education. Artificial Intelligence (AI) has the potential to address some of the biggest challenges in education today, innovate teaching and learning practices, and accelerate progress towards SDG 4. UNESCO is committed to supporting Member States to harness the potential of AI technologies for achieving the Education 2030 Agenda, while ensuring that its application in educational contexts is guided by the core principles of inclusion and equity. Within the framework of the\xa0Beijing Consensus, UNESCO developed\xa0Artificial intelligence and education: Guidance for policy-makers to foster the readiness of education policy-makers in artificial intelligence. Generative AI an

In [19]:
@tool
def fetch_url(url: str) -> str:
    """Fetch readable text content from a URL."""

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code != 200:
            return f"Failed to fetch page: {url}"

        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()

        text = soup.get_text(separator=" ", strip=True)

        return text[:12000]

    except Exception as e:
        return f"Error fetching {url}: {str(e)}"

In [20]:
test_url = "https://en.wikipedia.org/wiki/Artificial_intelligence_in_education"

print(fetch_url.invoke({"url": test_url})[:1000])

Artificial intelligence in education - Wikipedia Jump to content Main menu Main menu move to sidebar hide Navigation Main page Contents Current events Random article About Wikipedia Contact us Contribute Help Learn to edit Community portal Recent changes Upload file Special pages Search Search Appearance Donate Create account Log in Personal tools Donate Create account Log in Contents move to sidebar hide (Top) 1 History 2 Theory Toggle Theory subsection 2.1 Three paradigms of AIEd 2.2 Socio-technical imaginaries 2.3 Post-humanist and new materialist perspectives 3 Applications Toggle Applications subsection 3.1 AI-based tutoring systems 3.2 Generative AI 3.3 Emotional AI 4 Perspectives Toggle Perspectives subsection 4.1 Commercial perspectives 4.2 Institutional perspectives 4.3 Student perspectives 5 Problems Toggle Problems subsection 5.1 Over-reliance, inaccuracy, and academic integrity 5.2 Accessibility 5.3 Bias 5.4 Data privacy and intellectual property 5.5 Invisible labour and en

In [21]:
model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

In [22]:
response = model.invoke("Say hello in one short sentence.")
print(response.content)

Hello!


In [25]:
system_prompt = """
You are an education research agent.

You must follow this exact process:
1. Read the user's question carefully.
2. Use internet_search to find relevant information about AI in education.
3. Select exactly the top 3 URLs from the search results.
4. Call fetch_url for each of those 3 URLs at most once.
5. Use only successfully fetched page contents in the final answer.
6. If one URL fails, skip it and continue with the remaining successful sources.
7. Do not retry the same failed URL repeatedly.
8. Do not cite any URL that failed to fetch successfully.
9. Write a clear answer about the benefits, risks, and practical impact of AI in education.
10. Include only the URLs that were fetched successfully in the final answer.
"""

In [26]:
agent = create_agent(
    model=model,
    tools=[internet_search, fetch_url],
    system_prompt=system_prompt,
)

In [27]:
question = "What are the benefits and risks of artificial intelligence in education?"

response = agent.invoke({
    "messages": [
        {"role": "user", "content": question}
    ]
})

print(response["messages"][-1].content)

**Benefits of AI in Education**

| Benefit | How AI Helps | Source |
|--------|--------------|--------|
| **Personalized learning** – AI adapts content, pace, and feedback to each student’s strengths and weaknesses, creating custom learning paths. | Adaptive learning systems and intelligent tutoring platforms adjust lessons in real‑time, helping students master material at their own speed. | <https://www.ucanwest.ca/blog/education-careers-tips/advantages-and-disadvantages-of-ai-in-education> |
| **Improved accessibility** – Real‑time translation, speech‑to‑text, captioning, and other accommodations make learning inclusive for students with disabilities and multilingual learners. | AI‑driven tools provide instant language support and adaptive materials that level the playing field. | <https://www.ucanwest.ca/blog/education-careers-tips/advantages-and-disadvantages-of-ai-in-education> |
| **Administrative efficiency** – Automation of grading, scheduling, record‑keeping, and quiz creation